In [1]:
!pip install roboflow
from roboflow import Roboflow
rf = Roboflow(api_key="ys8HpBLUxOd3R4iYGsuB")
project = rf.workspace("glacierlake").project("glacier-lake")
version = project.version(8)
dataset = version.download("yolov8")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 92.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 133.1 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.12.0.88
    Uninstalling opencv-python-headless-4.12.0.88:
      Successfully uninstalled opencv-python-headless-4.12.0.88
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to glacier-lake-8 in yolov8:: 100%|██████████| 937/937 [00:00<00:00, 2241.79it/s]


In [2]:
!pip install ultralytics scikit-learn joblib --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 34.7 MB/s eta 0:00:00


In [3]:
import os
import cv2
import numpy as np
from ultralytics import YOLO
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import joblib
import glob
from tqdm import tqdm
import random

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [5]:
model = YOLO("bestglacier.pt")
print("YOLO model loaded successfully")

YOLO model loaded successfully


In [6]:
def extract_features_from_pair(img1_path, img2_path, yolo_model):
    def get_mask(img_path):
        results = yolo_model.predict(source=img_path, verbose=False)
        img = cv2.imread(img_path)
        mask = np.zeros(img.shape[:2], dtype=np.uint8)

        for r in results:
            if len(r.masks) > 0:
                for c in r:
                    contour = c.masks.xy.pop().astype(np.int32).reshape(-1, 1, 2)
                    cv2.drawContours(mask, [contour], -1, 255, thickness=cv2.FILLED)

        return mask

    mask1 = get_mask(img1_path)
    mask2 = get_mask(img2_path)

    if mask1.shape != mask2.shape:
        mask2 = cv2.resize(mask2, (mask1.shape[1], mask1.shape[0]), interpolation=cv2.INTER_NEAREST)

    area1 = np.sum(mask1 == 255)
    area2 = np.sum(mask2 == 255)
    intersection = np.sum(cv2.bitwise_and(mask1, mask2) == 255)
    union = np.sum(cv2.bitwise_or(mask1, mask2) == 255)

    if area1 == 0:
        return None

    growth_percentage = ((area2 - area1) / area1) * 100
    area_ratio = area2 / area1 if area1 > 0 else 0
    overlap_ratio = intersection / area1 if area1 > 0 else 0
    iou = intersection / union if union > 0 else 0

    contours1, _ = cv2.findContours(mask1, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours2, _ = cv2.findContours(mask2, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    perimeter1 = sum([cv2.arcLength(c, True) for c in contours1])
    perimeter2 = sum([cv2.arcLength(c, True) for c in contours2])

    compactness1 = (4 * np.pi * area1) / (perimeter1 ** 2) if perimeter1 > 0 else 0
    compactness2 = (4 * np.pi * area2) / (perimeter2 ** 2) if perimeter2 > 0 else 0

    features = [
        area1,
        area2,
        growth_percentage,
        area_ratio,
        overlap_ratio,
        iou,
        perimeter1,
        perimeter2,
        compactness1,
        compactness2
    ]

    return features

print("Feature extraction function defined")

Feature extraction function defined


In [7]:
dataset_path = "glacier-lake-8/train/images"
image_files = glob.glob(os.path.join(dataset_path, "*.jpg"))

print(f"Found {len(image_files)} images")

X = []
y = []

random.shuffle(image_files)

for i in tqdm(range(len(image_files) - 1)):
    img1 = image_files[i]
    img2 = image_files[i + 1]

    features = extract_features_from_pair(img1, img2, model)

    if features is not None:
        growth_pct = features[2]

        if growth_pct > 15:
            label = 1
        else:
            label = 0

        X.append(features)
        y.append(label)

X = np.array(X)
y = np.array(y)

print(f"Generated {len(X)} training samples")
print(f"Risky samples: {np.sum(y == 1)}, Not Risky samples: {np.sum(y == 0)}")

Found 423 images


100%|██████████| 422/422 [00:39<00:00, 10.62it/s]

Generated 422 training samples
Risky samples: 168, Not Risky samples: 254


In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)

rf_model.fit(X_train, y_train)

train_score = rf_model.score(X_train, y_train)
test_score = rf_model.score(X_test, y_test)

print(f"Training Accuracy: {train_score * 100:.2f}%")
print(f"Testing Accuracy: {test_score * 100:.2f}%")

Training Accuracy: 100.00%
Testing Accuracy: 98.82%


In [9]:
y_pred = rf_model.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Not Risky', 'Risky']))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

feature_names = ['area1', 'area2', 'growth_pct', 'area_ratio', 'overlap_ratio', 'iou', 'perimeter1', 'perimeter2', 'compactness1', 'compactness2']
importances = rf_model.feature_importances_

print("\nFeature Importances:")
for name, imp in sorted(zip(feature_names, importances), key=lambda x: x[1], reverse=True):
    print(f"{name}: {imp:.4f}")

Classification Report:
              precision    recall  f1-score   support

   Not Risky       0.98      1.00      0.99        51
       Risky       1.00      0.97      0.99        34

    accuracy                           0.99        85
   macro avg       0.99      0.99      0.99        85
weighted avg       0.99      0.99      0.99        85


Confusion Matrix:
[[51  0]
 [ 1 33]]

Feature Importances:
growth_pct: 0.4299
area_ratio: 0.3990
overlap_ratio: 0.0520
area2: 0.0441
perimeter2: 0.0254
area1: 0.0225
perimeter1: 0.0164
iou: 0.0074
compactness1: 0.0020
compactness2: 0.0013


In [10]:
joblib.dump(rf_model, 'risk_classifier.pkl')
print("Model saved as 'risk_classifier.pkl'")
print("Download this file and use it in your Streamlit app")

Model saved as 'risk_classifier.pkl'
Download this file and use it in your Streamlit app


In [11]:
!pip install xgboost --quiet
import xgboost as xgb
from sklearn.ensemble import VotingClassifier
print("XGBoost imported successfully")

XGBoost imported successfully


In [12]:
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)

xgb_model.fit(X_train, y_train)

xgb_train_score = xgb_model.score(X_train, y_train)
xgb_test_score = xgb_model.score(X_test, y_test)

print(f"XGBoost Training Accuracy: {xgb_train_score * 100:.2f}%")
print(f"XGBoost Testing Accuracy: {xgb_test_score * 100:.2f}%")

XGBoost Training Accuracy: 100.00%
XGBoost Testing Accuracy: 98.82%


In [13]:
hybrid_model = VotingClassifier(
    estimators=[
        ('random_forest', rf_model),
        ('xgboost', xgb_model)
    ],
    voting='soft'
)

hybrid_model.fit(X_train, y_train)

hybrid_train_score = hybrid_model.score(X_train, y_train)
hybrid_test_score = hybrid_model.score(X_test, y_test)

print(f"Hybrid Model Training Accuracy: {hybrid_train_score * 100:.2f}%")
print(f"Hybrid Model Testing Accuracy: {hybrid_test_score * 100:.2f}%")

Hybrid Model Training Accuracy: 100.00%
Hybrid Model Testing Accuracy: 98.82%


In [14]:
y_pred_hybrid = hybrid_model.predict(X_test)

print("Hybrid Model Classification Report:")
print(classification_report(y_test, y_pred_hybrid, target_names=['Not Risky', 'Risky']))

print("\nHybrid Model Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_hybrid))

print("\n" + "="*50)
print("MODEL COMPARISON:")
print("="*50)
print(f"Random Forest Accuracy: {test_score * 100:.2f}%")
print(f"XGBoost Accuracy: {xgb_test_score * 100:.2f}%")
print(f"Hybrid Model Accuracy: {hybrid_test_score * 100:.2f}%")

Hybrid Model Classification Report:
              precision    recall  f1-score   support

   Not Risky       0.98      1.00      0.99        51
       Risky       1.00      0.97      0.99        34

    accuracy                           0.99        85
   macro avg       0.99      0.99      0.99        85
weighted avg       0.99      0.99      0.99        85


Hybrid Model Confusion Matrix:
[[51  0]
 [ 1 33]]

MODEL COMPARISON:
Random Forest Accuracy: 98.82%
XGBoost Accuracy: 98.82%
Hybrid Model Accuracy: 98.82%


In [15]:
joblib.dump(hybrid_model, 'hybrid_risk_model.pkl')
joblib.dump(rf_model, 'random_forest_model.pkl')
joblib.dump(xgb_model, 'xgboost_model.pkl')

print("✓ Hybrid model saved as: hybrid_risk_model.pkl")
print("✓ Random Forest saved as: random_forest_model.pkl")
print("✓ XGBoost saved as: xgboost_model.pkl")
print("\nDownload 'hybrid_risk_model.pkl' for your Streamlit app")

✓ Hybrid model saved as: hybrid_risk_model.pkl
✓ Random Forest saved as: random_forest_model.pkl
✓ XGBoost saved as: xgboost_model.pkl

Download 'hybrid_risk_model.pkl' for your Streamlit app


In [16]:
import sys
import sklearn
print(f"Python: {sys.version}")
print(f"NumPy: {np.__version__}")
print(f"OpenCV: {cv2.__version__}")
print(f"scikit-learn: {__import__('sklearn').__version__}")
print(f"XGBoost: {xgb.__version__}")
print(f"Joblib: {joblib.__version__}")
print(f"Ultralytics: {__import__('ultralytics').__version__}")
print(f"Roboflow: {__import__('roboflow').__version__}")

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
NumPy: 2.0.2
OpenCV: 4.10.0
scikit-learn: 1.6.1
XGBoost: 3.1.2
Joblib: 1.5.3
Ultralytics: 8.3.243
Roboflow: 1.2.11
